In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", device_map="auto")

d:\All_Coding_stuff\Arnav Projects\Adaptive LLM inference and routing\.AdaptiveLLM\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mishr\.cache\huggingface\hub\models--Qwen--Qwen3-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 6079.15it/s]


In [17]:
prompt = """You are a model-selection router. Analyze the user's input and select exactly one model.

MODELS:
- local-model: fast, cheap, runs locally. Good for simple writing, rewriting, summarization, formatting, basic coding, classification.
- cloud-glm: slower, costs more, accessed via API. Use for deep reasoning, multi-step analysis, current/niche information, research synthesis, or when accuracy is critical.

RULES:
- Default to local-model unless the task clearly requires cloud-glm's strengths.
- If uncertain, choose local-model.

EXAMPLES:
Input: "Fix the grammar in this paragraph."
Output: local-model

Input: "Compare the economic policies of three countries and explain which approach worked best, citing recent data."
Output: cloud-glm

Input: "Write a Python function to reverse a string."
Output: local-model

Input: "What are the latest developments in quantum computing this year?"
Output: cloud-glm

OUTPUT FORMAT:
Respond with exactly one token: local-model OR cloud-glm
No punctuation, no explanation, no extra text.

USER INPUT:
{user_input}
"""

In [19]:
import copy
import torch

router_prompt = prompt  # already defined by you

# ---- Build and cache the static prefix once ----
prefix_messages = [{"role": "system", "content": router_prompt}]
prefix_ids = tokenizer.apply_chat_template(
    prefix_messages,
    add_generation_prompt=False,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    prefix_out = model(prefix_ids["input_ids"], use_cache=True)
base_cache = prefix_out.past_key_values  # reusable KV cache


def route(user_input: str, max_new_tokens: int = 10) -> str:
    messages = [
        {"role": "system", "content": router_prompt},
        {"role": "user", "content": user_input},
    ]
    ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(ids["input_ids"], max_new_tokens=max_new_tokens)

    return tokenizer.decode(
        output[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True
    ).strip()


# ---- Test it ----
print(route("My name is Arnav Mishra, what is the greatest book on this entire planet"))
print(route("Summarize this paragraph in one sentence: The economy grew by 3% last quarter."))
print(route("help me with this equation of the black hole"))

local-model
local-model
local-model


In [20]:
print(route("What's the latest news on the Fed's interest rate decision this month?"))
print(route("Compare the GDP growth of Japan, Germany, and India over the last decade and explain the key economic drivers."))

cloud-glm
cloud-glm


In [21]:
print(route("My name is Arnav Mishra, what is the greatest book on this entire planet"))
print(route("Summarize this paragraph in one sentence: The economy grew by 3% last quarter."))

local-model
local-model
